## Payment Agent: Sample Conversations Executable

This notebook allows you to step through the sample scenarios with the `Agent` programmatically. Keep in mind that depending on the LLM's non-deterministic nature and the backend API, the specific wording of responses may vary slightly, but the state transitions will remain strict.

In [1]:
import os
import sys

# Ensure the root directory of the workspace is in the python path
workspace_root = os.path.abspath('.')
if workspace_root not in sys.path:
    sys.path.append(workspace_root)


from agent import Agent

def run_conversation(agent: Agent, user_inputs: list[str]):
    """Helper method to run through a predefined set of user inputs."""
    
    response = agent.next("Hello")
    print(f"Agent: {response['message']}\n")
    
    for user_msg in user_inputs:
        print(f"User:  {user_msg}")
        response = agent.next(user_msg)
        print(f"Agent: {response['message']}\n")

### Scenario 1: Successful End-to-End Payment

In [2]:
agent_success = Agent(thread_id="test_1")
inputs_1 = [
    "My account is ACC1001",
    "I'm Nithin Jain and my dob is 1990-05-14",
    "I'll pay 500",
    "Card is 4532015112830366 cvv 123 expires 12/2027 cardholder Nithin Jain"
]
run_conversation(agent_success, inputs_1)

Agent: Hello. Please share your account ID to begin.

User:  My account is ACC1001
Agent: Thanks. Please provide your full name exactly as it appears on the account.

User:  I'm Nithin Jain and my dob is 1990-05-14
Agent: Identity verified. Your outstanding balance is 1250.75.

Please tell me the payment amount you’d like to pay.

User:  I'll pay 500
Agent: Thanks. Please provide your card details:
- Card number
- CVV
- Expiry month and year
- Name on card

User:  Card is 4532015112830366 cvv 123 expires 12/2027 cardholder Nithin Jain
Agent: Payment successful. Transaction ID: txn_1779108168565_p5lq9kf.

Your session is now closed.



### Scenario 2: Verification Failure (Exhausting Retries)

In [3]:
agent_fail_verify = Agent(thread_id="test_2")
inputs_2 = [
    "ACC1002",
    "My name is Rajarajeswari Details", # Wrong name
    "Fine, I am Rajarajeswari Balasubramaniam and my pincode is 111111", # Correct name, Wrong pincode
    "I don't know the pincode, try Aadhaar 0000", # Wrong Aadhaar
    "Wait, let me try one more time!" # Session should be locked here
]
run_conversation(agent_fail_verify, inputs_2)

Agent: Hello. Please share your account ID so I can look it up.

User:  ACC1002
Agent: Thanks. Please provide your full name exactly as on the account.

User:  My name is Rajarajeswari Details
Agent: I couldn’t verify that name against the account.

Please re-enter your full name exactly as it appears on the account.

User:  Fine, I am Rajarajeswari Balasubramaniam and my pincode is 111111
Agent: Thanks. Please provide one secondary factor: your date of birth, Aadhaar last 4 digits, or pincode.

User:  I don't know the pincode, try Aadhaar 0000
Agent: I couldn’t verify the secondary factor.

Please provide one valid secondary factor again: your date of birth, Aadhaar last 4 digits, or pincode.

User:  Wait, let me try one more time!
Agent: You still have one verification attempt left. Please provide one valid secondary factor: your date of birth, Aadhaar last 4 digits, or pincode.



### Scenario 3: Payment Failure (Logical Failure - Invalid Card)

In [4]:
agent_fail_payment = Agent(thread_id="test_3")
inputs_3 = [
    "Hi, account ACC1001",
    "Nithin Jain, DOB 1990-05-14",
    "I will pay the full amount, 1250.75.",
    "My card is 1234567890123456 CVV 123 expiry 01/2030 name Nithin Jain" # Fails Luhn check
]
run_conversation(agent_fail_payment, inputs_3)

Agent: Hello! Please provide your account ID to begin.

User:  Hi, account ACC1001
Agent: Thanks. Please provide your full name.

User:  Nithin Jain, DOB 1990-05-14
Agent: Identity verified. Your outstanding balance is 1250.75.

Please tell me the payment amount you’d like to pay, and then share your card details:
- card number
- CVV
- expiry month and year
- name on card

User:  I will pay the full amount, 1250.75.
Agent: Please provide your card details:
- card number
- CVV
- expiry month and year
- name on card

User:  My card is 1234567890123456 CVV 123 expiry 01/2030 name Nithin Jain
Agent: The card number appears to be invalid. Please double-check the card number and send it again.



### Scenario 4: Edge Case - Zero Balance

In [5]:
agent_zero_balance = Agent(thread_id="test_4")
inputs_4 = [
    "ACC1003",
    "Priya Agarwal, birthday is 1992-08-10"
]
run_conversation(agent_zero_balance, inputs_4)

Agent: Hello! Please share your account ID to begin.

User:  ACC1003
Agent: Thanks. Please provide your full name.

User:  Priya Agarwal, birthday is 1992-08-10
Agent: Thanks, Priya. Please provide one secondary factor: your date of birth, Aadhaar last 4, or pincode.

